In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [ ]:
import os
from typing import Literal
from deepagents import create_deep_agent
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
        query:str,
        max_results: int = 5,
        topic: Literal["general","news","finance"]="general",
        include_raw_content: bool = False,
):
    """Run a web search """
    return tavily_client.search(
        query,
        max_results = max_results,
        include_raw_content = include_raw_content,
        topic = topic,
    )

research_subagent = {
    "name" : "research-agent",
    "description" : "Used to research more in depth questions",
    "system_prompt" : "You are a great researcher",
    "tools":[internet_search],
    "model":"groq:openai/gpt-oss-20b"
}

subagents = [research_subagent]

agent = create_deep_agent(
    model="groq:openai/gpt-oss-20b",
    subagents=subagents,
)
agent

In [ ]:
result = agent.invoke(
    {
        "messages":[{
            "role":"user",
            "content":"How do I build a langgraph with conditional routing and memory? Show a minimal example."
        }],
    },
    config = {"configurable":{'thread_id':"skills-domo-2"}},
)

for m in result["messages"]:
    m.pretty_print()

## Structured Output with subagents


In [ ]:
from pydantic import BaseModel, Field
from deepagents import create_deep_agent

class ResearchFinding(BaseModel):
    """Strucured findings from a research task """
    summary : str = Field(description="Summary of findings")
    confidence : float = Field(description="Confidence score from 0 to 1")
    sources: list[str] = Field(description="List of source URLs")


research_subagent = {
    "name":"researcher",
    "description":"Researches topics and returns structured findings",
    "system_prompt":"Research the given topic thoroughly.Return your findings",
    "tools":[internet_search],
  
}

agent = create_deep_agent(
    model = "groq:openai/gpt-oss-20b",
    subagents=[research_subagent],
)

result = await agent.invoke(
    {"messages":[{"role":"user","content":"Research Finding recent in quantum computing"}]}
)

In [ ]:
result